# SLICE-FPGA-II control: CEO lock, GPS clock, monitoring

This notebook replaces Vescent's PyQt GUI (`XEM_GUI3.py`) and runs on **Python 3.9 to 3.13 with NumPy 2**. It uses the `slicefpga` package in this folder:

| package piece | replaces |
|---|---|
| `slicefpga.rp_pll` | `RP_PLL.py`: TCP protocol to `monitor-tcp` on the box (port 5000), now thread-safe |
| `slicefpga.superlaserland` | `SuperLaserLand_JD_RP.py` + `SuperLaserLand2_JD2_PLL.py`: register map, unchanged values |
| `slicefpga.controller.SliceFPGA` | the GUI windows' logic: push defaults, clock, lock/unlock, attach |
| `slicefpga.monitor` | Counters tab + Auto Recover/Unlock, logging to CSV, fixed-size memory |

**Equivalence.** `tests/verify_against_gui.py` replays push-defaults, external clock, CEO lock/unlock and optical lock/unlock. It does this against a recording of the original GUI (Python 3.9 / PyQt5) that was captured on a fake box. Every register ends in the same state. The only exception is that the GUI writes placeholder PLL1 gains while the optical loop is open; those converge at the first optical lock. Clock, lock and unlock steps write the same registers in the same order.

**No GUI memory leak.** The old GUI slowed down over hours because its small IQ scatter plot adds two entries to pyqtgraph's symbol cache on every refresh, forever. Here there is no live plot. History lives in a fixed-length buffer, and plots are drawn only when you run a cell.

**The lock lives in the FPGA.** The loop filters run in the FPGA fabric. Closing this notebook or calling `fpga.close()` writes nothing, so the box stays locked. Use `attach()` to reconnect without disturbing it.

Network: give the PC's Ethernet adapter a static address `192.168.0.x` (not `.150`). The box is `192.168.0.150`.

## 0. Settings

In [ ]:
# ---- edit these ----------------------------------------------------------------
SIMULATE   = False          # True: dry-run against the built-in software box (no hardware needed)
HOST       = '192.168.0.150'       # SLICE-FPGA-II default address
PORT       = 5000
CONFIG     = None                  # None -> slicefpga/configs/UCF_FFC-100_Parameters.xml (your unit's file)
F_EXT      = 20e6                  # EXT CLK input: GPS 10 MHz through the doubler = 20 MHz  -> fs = 125.000000 MHz exactly
LOG_DIR    = 'slice_logs'          # CSV logs from the monitor
# ---------------------------------------------------------------------------------
import json, logging, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import vescentcomb
slicefpga = vescentcomb.addon("comb_lock").fpga          # the SliceFPGA_II_modern package, built into the comb lock module
from vescentcomb_addons.comb_lock.fpga import SliceFPGA, SliceConfig, Monitor, Watchdog, discover, pll_multipliers

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')
logging.getLogger('vescentcomb_addons.comb_lock.fpga.superlaserland').setLevel(logging.WARNING)   # silence the vendored debug chatter

if SIMULATE:
    from vescentcomb_addons.comb_lock.fpga.simulator import SliceSimulator
    sim = SliceSimulator(port=50000).start()
    HOST, PORT = '127.0.0.1', 50000
print('slicefpga', slicefpga.__version__, '| target', f'{HOST}:{PORT}', '| SIMULATE =', SIMULATE)

## 1. Find the box (optional)
UDP broadcast discovery, the same as the GUI's "Broadcast discovery" button. If nothing answers, check the static IP and the firewall rule for `python.exe`.

In [ ]:
if not SIMULATE:
    print(discover('192.168.0.255', timeout=2.0))   # -> [('192.168.0.150', '<MAC>')]; UCF unit MAC 00:26:32:f0:a9:6b

## 2. Clock plan check (no hardware)
The FPGA synthesizes its ADC clock from the EXT CLK input with an MMCM. 20 MHz in gives MULT = 50 and DIV = 8, so the VCO runs at 1000 MHz and **fs = 125.000000 MHz**. A plain 10 MHz would give 124 MHz.

In [ ]:
for f in (10e6, 20e6):
    M, D, D1 = pll_multipliers(f)
    print(f'{f/1e6:5.1f} MHz in -> MULT {M:2d}, DIV {D}: VCO {f*M/1e6:6.1f} MHz, fs = {f*M/D/1e6:.6f} MHz')

## 3. Connect

Pick **one** of the two cells below:
* **3a, configure from scratch.** Use this after a power cycle. It does the GUI's *Push default values* from your XML, then switches to the 20 MHz external clock. Both loops are left **open**.
* **3b, attach.** Use this when the box is already running or locked. It reads the clock, reference frequencies, offsets and gains back from the FPGA and **writes nothing**.

In [ ]:
# 3a: configure from scratch
fpga = SliceFPGA(HOST, PORT, config=CONFIG)
fpga.push_config(clock=('external', F_EXT))
pd.Series(fpga.status()).to_frame('value')

In [ ]:
# 3b: attach to a box that is already running (read-only). Uncomment to use instead of 3a.
# fpga = SliceFPGA(HOST, PORT, config=CONFIG)
# pd.Series(fpga.attach(f_ext=F_EXT)).to_frame('value')

### Sanity checks
* `clock` should be **external** and `ext_clk_measured_MHz` should be about 20.0000. That readout is taken against the Zynq's own undisciplined 200 MHz, so a few ppm off is normal. A gross error means the doubler output is dirty or too weak (it needs about 0 dBm and a clean 20 MHz).
* `ddc0_ref_MHz` is the CEO lock point. Its sign is negative because the GUI default is "VCO sign +".

In [ ]:
cfg = fpga.config
rows = {
  'CEO ref [MHz]':         cfg.getf('Reference_frequency', 'DDC0') / 1e6,
  'CEO VCO gain [Hz/V]':   cfg.getf('VCO_gain', 'DAC0'),
  'PLL0 kp [dB]':          cfg.getf('PLL0_settings', 'kp'),
  'PLL0 fi [Hz]':          cfg.getf('PLL0_settings', 'fi'),
  'PLL0 fii [Hz]':         cfg.getf('PLL0_settings', 'fii'),
  'PLL0 fd / fdf [Hz]':    f"{cfg.getf('PLL0_settings', 'fd'):.3g} / {cfg.getf('PLL0_settings', 'fdf'):.3g}",
  'DAC0 limits [V]':       f"{cfg.getf('Output_limits_low', 'DAC0')} .. {cfg.getf('Output_limits_high', 'DAC0')}",
  'DAC0 offset [V]':       cfg.getf('Output_offset_in_volts', 'DAC0'),
  'fs [MHz]':              fpga.sl.fs / 1e6,
}
g = fpga.loop_gains('ceo', lock=True)
rows.update({'P (counts/count)': g.P, 'I': g.I, 'II': g.II, 'D': g.D})
pd.Series(rows).to_frame('CEO loop')

**f_ceo lock point.** The UCF config sets the DDC reference to **21.4 MHz**. If your f_ceo setpoint is 21.6 MHz, change it here before locking. This updates the in-memory config and the FPGA; use `fpga.config.save(...)` to keep it.

In [ ]:
# fpga.set_ref_freq('ceo', 21.6e6)        # uncomment to change the CEO lock point
print('CEO DDC reference: %.6f MHz' % (fpga.sl.get_ddc0_ref_freq_from_RAM() / 1e6))

## 4. Look at the CEO beat before locking
A one-shot capture of ADC0 (16k samples at fs). The beat should sit at the DDC reference frequency. The figure is closed after it is drawn, so repeated runs do not pile up plot objects.

In [ ]:
x, _ = fpga.capture('ADC0', 2**14)
x = x - x.mean()
w = np.blackman(len(x))
spec = np.abs(np.fft.rfft(x * w))**2
f = np.fft.rfftfreq(len(x), 1 / fpga.sl.fs)
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(f / 1e6, 10 * np.log10(spec / spec.max() + 1e-15))
ax.axvline(abs(fpga.sl.get_ddc0_ref_freq()) / 1e6, color='r', ls='--', lw=1, label='DDC0 reference')
ax.set_xlabel('MHz'); ax.set_ylabel('dB (rel.)'); ax.set_ylim(-90, 3); ax.legend(); ax.set_title('ADC0 (f_ceo beat) spectrum')
plt.show(); plt.close(fig)
print('peak at %.4f MHz' % (f[np.argmax(spec)] / 1e6))

## 5. Lock the CEO
This is equivalent to ticking **Lock** on the CEO tab: automatic dither off, then PLL0 gains plus the lock bit. A few counter reads follow so you can confirm it is locked. `ceo_err_Hz` is the in-loop frequency error from the zero-dead-time counter (about a 1 s gate).

In [ ]:
fpga.lock('ceo')
time.sleep(0.5)
rows = []
for _ in range(8):
    s = fpga.read_counters()
    if s['new']:
        rows.append(s)
    time.sleep(0.4)
print('CEO lock bit in FPGA:', fpga.is_locked('ceo'))
pd.DataFrame(rows)[['n', 'ceo_err_Hz', 'dac0_V', 'dac0_frac']]

## 6. Monitor, log and auto-relock
A background thread polls the counters (period 0.25 s; a new sample arrives each gate), appends to `LOG_DIR/slice_fpga_<time>.csv`, and keeps the last `history` samples in RAM.

`Watchdog` options:
* `action='log'` only records an event.
* `'unlock'` behaves like the GUI's Auto Unlock.
* `'relock'` works like Auto Recover. It opens the loop, puts the manual offset back where the lock last sat, then relocks. It is capped at `max_relocks` per `window_s`; after that it leaves the loop open and switches to log-only, so a dead beat is not hammered forever.

A channel is "railed" when its DAC is within `rail_margin` (fraction of range) of a limit, or the error exceeds `max_err_Hz`, for `n_bad` consecutive samples.

The relock path has only been tested against the simulator. Try it once on the bench, for example by blocking the beat, before you trust it overnight.

In [ ]:
mon = Monitor(fpga, csv_dir=LOG_DIR, period_s=0.25, history=20000,
              watchdog=Watchdog('ceo', action='relock', rail_margin=0.05, max_err_Hz=None, n_bad=2,
                                max_relocks=5, window_s=3600))
mon.start()
time.sleep(5)
print('logging to', mon.csv_path)
mon.summary(last_s=60)

In [ ]:
mon.plot(last_s=600)         # re-run any time; the figure is closed after drawing

In [ ]:
pd.DataFrame([(time.strftime('%H:%M:%S', time.localtime(t)), m) for t, m in mon.events], columns=['time', 'event'])

## 7. CEO frequency / phase noise (on demand)
This is what the GUI's phase-noise panel showed. It captures 32k samples of the DDC0 instantaneous-frequency stream, decimates by 10, and takes a Welch PSD.

In [ ]:
from scipy.integrate import trapezoid
fr, S_nu, S_phi = fpga.frequency_noise('ceo', n=2**15)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].loglog(fr, S_nu); ax[0].set_xlabel('Hz'); ax[0].set_ylabel('S_nu [Hz^2/Hz]'); ax[0].set_title('frequency noise')
ax[1].semilogx(fr, 10 * np.log10(S_phi / 2)); ax[1].set_xlabel('Hz'); ax[1].set_ylabel('L(f) [dBc/Hz]'); ax[1].set_title('phase noise')
plt.tight_layout(); plt.show(); plt.close(fig)
band = (fr > 100) & (fr < fr[-1])
print('integrated phase noise %.0f Hz .. %.0f kHz: %.3g rad rms' % (fr[band][0], fr[band][-1] / 1e3, np.sqrt(trapezoid(S_phi[band], fr[band]))))

## 8. Save / restore the lock setup
A snapshot holds the full XML config as currently edited, the manual DAC offsets, the VCO sign, the clock and the lock state. It is plain JSON.

In [ ]:
snap = fpga.snapshot()
Path('slice_snapshot.json').write_text(json.dumps(snap, indent=1))
fpga.config.save('UCF_FFC-100_Parameters_current.xml')       # an XML the old GUI can also import
print('saved snapshot', snap['time'], '| locked:', snap['locked'])

In [ ]:
# After a power cycle: push everything back and relock as saved. Uncomment to use.
# fpga.restore(json.loads(Path('slice_snapshot.json').read_text()), relock=True)

## 9. Leave it running without the laptop
Stopping the monitor and closing the socket writes nothing to the box. The CEO loop keeps running in the FPGA.

What stops when you do this: watchdog relocks and logging. Later, reconnect with **3b, `attach()`**, which also writes nothing, and check the counter and DAC to confirm the lock held.

In [ ]:
mon.stop()
fpga.close()
print('disconnected; FPGA left as-is')

In [ ]:
# Re-attach later (read-only) and check the lock held:
fpga = SliceFPGA(HOST, PORT, config=CONFIG)
st = fpga.attach(f_ext=F_EXT)
print({k: st[k] for k in ('clock', 'fs_MHz', 'ceo_locked', 'ddc0_ref_MHz', 'dac0_frac')})
print(fpga.read_counters())

## 10. Optical lock (f_opt), if used
This follows the same procedure as the GUI's Optical tab. First it grabs the beat with the slow DAC2 integrator, then after 0.2 s it closes PLL1 on DAC1 and holds the slow integrator.

In [ ]:
# fpga.lock('optical')
# fpga.unlock('optical')
print('optical locked:', fpga.is_locked('optical'), '| mode:', fpga.optical_mode)

## 11. Unlock and finish

In [ ]:
fpga.unlock('ceo')            # ramps the manual offset to where the lock sat, then opens the loop (as the GUI does)
print('CEO locked:', fpga.is_locked('ceo'))
fpga.close()
if SIMULATE:
    sim.stop()

---
### Reference
* `fpga.set_loop_filter('ceo', kp=-60, fi=3e4)` changes the gains (kp in dB, corners in Hz), keeping the lock state.
* `fpga.set_vco_gain(0, 4e3)` sets the actuator gain used to normalize the loop gains.
* `fpga.set_dac_offset(0, 0.1)` sets the manual DAC0 offset in the GUI's volts.
* `fpga.set_clock(True, 20e6)` or `fpga.set_clock(False)` selects the external or internal ADC clock.
* `fpga.capture('ADC0'|'ADC1'|'DDC0'|'DDC1'|'DAC0'|'DAC1'|'DAC2', n)` returns raw logger blocks (max 32768).
* `fpga.program_fpga()` reloads `red_pitaya_top_extint_clk.bit` (as the GUI's "Program FPGA"). You only need this if the box lost its bitstream.
* Run `python tests/verify_against_gui.py tests/rec_gui_ucf.json` to repeat the GUI-equivalence check.